# Interop & Roundtripping — Q#, OpenQASM, JSON & Circuit Composition

This notebook demonstrates the cross-format capabilities of
[qdk-pythonic](https://github.com/brycewestheimer/qdk-pythonic):
generating Q# and OpenQASM 3.0 from Python circuits, parsing them
back, roundtripping through multiple formats, JSON serialization,
parameterized circuits, and circuit composition.

In [ ]:
%pip install qsharp>=1.25 git+https://github.com/brycewestheimer/qdk-pythonic.git

In [ ]:
from qdk_pythonic import Circuit, Parameter, bell_state, qft

print("Imports successful.")

---

## Code Generation

Every `Circuit` object can generate Q# via `to_qsharp()` and
OpenQASM 3.0 via `to_openqasm()`. The output is valid source code
ready for compilation.

In [ ]:
# Bell state
circ = bell_state(measure=True)

print("=== Q# ===")
print(circ.to_qsharp())
print()
print("=== OpenQASM 3.0 ===")
print(circ.to_openqasm())

In [ ]:
# QFT (more complex)
qft_circ = qft(4)
qft_circ.measure_all()

print("=== Q# (4-qubit QFT) ===")
print(qft_circ.to_qsharp())
print()
print("=== OpenQASM 3.0 (4-qubit QFT) ===")
print(qft_circ.to_openqasm())

---

## Parsing from Q#

`Circuit.from_qsharp()` parses a Q# code block back into a `Circuit`
object, recovering the gate structure, qubit allocations, and
measurements.

In [ ]:
qs_source = """
{
    use q = Qubit[2];
    H(q[0]);
    CNOT(q[0], q[1]);
    let r0 = MResetZ(q[0]);
    let r1 = MResetZ(q[1]);
    [r0, r1]
}
"""

parsed = Circuit.from_qsharp(qs_source)
print(f"Parsed: {parsed}")
print(f"Depth: {parsed.depth()}, Gates: {parsed.gate_count()}")
print()
print(parsed.draw())

In [ ]:
# More complex: Q# with controlled and adjoint gates
qs_complex = """
{
    use q = Qubit[3];
    H(q[0]);
    Controlled X([q[0]], q[1]);
    Rx(1.5707, q[2]);
    Adjoint S(q[1]);
    let r0 = MResetZ(q[0]);
    let r1 = MResetZ(q[1]);
    let r2 = MResetZ(q[2]);
    [r0, r1, r2]
}
"""

parsed_complex = Circuit.from_qsharp(qs_complex)
print(f"Parsed: {parsed_complex}")
print(f"Depth: {parsed_complex.depth()}, Gates: {parsed_complex.gate_count()}")

---

## Parsing from OpenQASM

`Circuit.from_openqasm()` parses OpenQASM 3.0 source strings.

In [ ]:
qasm_source = """
OPENQASM 3.0;
include "stdgates.inc";

qubit[3] q;

h q[0];
cx q[0], q[1];
cx q[1], q[2];

bit[3] c;
c[0] = measure q[0];
c[1] = measure q[1];
c[2] = measure q[2];
"""

parsed_qasm = Circuit.from_openqasm(qasm_source)
print(f"Parsed: {parsed_qasm}")
print(f"Depth: {parsed_qasm.depth()}")
print()
print(parsed_qasm.draw())

---

## Cross-Format Roundtrip

Build a circuit in Python, export to Q#, parse back, export to OpenQASM,
parse back again. Verify structural equivalence (depth, gate count) at
each step.

In [ ]:
# Step 1: Build original circuit
original = Circuit()
q = original.allocate(3)
original.h(q[0]).cx(q[0], q[1]).cx(q[1], q[2])

# Step 2: Python -> Q# -> Circuit
qs_code = original.to_qsharp()
from_qs = Circuit.from_qsharp(qs_code)

# Step 3: Circuit -> OpenQASM -> Circuit
qasm_code = from_qs.to_openqasm()
from_qasm = Circuit.from_openqasm(qasm_code)

# Verify at each step
print(f"{'Stage':<20} {'Qubits':>6} {'Gates':>5} {'Depth':>5}")
print("-" * 40)
for label, c in [
    ("Original", original),
    ("After Q# roundtrip", from_qs),
    ("After QASM roundtrip", from_qasm),
]:
    print(f"{label:<20} {c.qubit_count():>6} {c.total_gate_count():>5} {c.depth():>5}")

---

## JSON Serialization

Circuits can be serialized to JSON for storage, sharing, and
reconstruction. The JSON format preserves gate types, parameters
(including symbolic), qubit labels, and optional metadata.

In [ ]:
circ = bell_state(measure=True)
json_str = circ.to_json(name="bell_pair", metadata={"author": "demo", "purpose": "entanglement"})
print(json_str)

In [ ]:
reconstructed = Circuit.from_json(json_str)
print(f"Reconstructed: {reconstructed}")
print(f"Depth: {reconstructed.depth()}, Gates: {reconstructed.gate_count()}")
print(f"Structurally equal: {reconstructed == circ}")

---

## Parameterized Circuits

`Parameter` objects act as symbolic placeholders for rotation angles.
They serialize as `{"kind": "symbolic", "name": "..."}` in JSON and
must be bound with `bind_parameters()` before code generation.

In [ ]:
theta = Parameter("theta")
phi = Parameter("phi")

param_circ = Circuit()
q = param_circ.allocate(2)
param_circ.ry(theta, q[0])
param_circ.rz(phi, q[1])
param_circ.cx(q[0], q[1])

print(f"Parameters: {[p.name for p in param_circ.parameters]}")

In [ ]:
# JSON with symbolic parameters
param_json = param_circ.to_json(name="parameterized_example")
print(param_json)

In [ ]:
# Bind parameters and generate Q#
bound = param_circ.bind_parameters({"theta": 0.5, "phi": 1.2})

print("=== Before binding (error expected) ===")
try:
    param_circ.to_qsharp()
except Exception as e:
    print(f"  {type(e).__name__}: {e}")

print()
print("=== After binding ===")
print(bound.to_qsharp())

---

## Circuit Composition

Circuits can be combined using `compose_into()` (append with qubit
mapping) and `+` (concatenation into a new circuit with fresh
qubit allocations).

In [ ]:
# compose_into with positional mapping
main = Circuit()
q = main.allocate(3)
main.h(q[0]).h(q[1]).h(q[2])

sub = Circuit()
sq = sub.allocate(3)
sub.cx(sq[0], sq[1]).cx(sq[1], sq[2])

# Appends sub's instructions to main using positional mapping
main.compose_into(sub)

print("After compose_into (positional):")
print(f"  Qubits: {main.qubit_count()}, Gates: {main.total_gate_count()}")
print(main.draw())

In [ ]:
# compose_into with explicit qubit_map
target = Circuit()
tq = target.allocate(4)
target.h(tq[0])

sub2 = Circuit()
sq2 = sub2.allocate(2)
sub2.cx(sq2[0], sq2[1])

# Map sub2's qubits to specific positions in target
target.compose_into(sub2, qubit_map={sq2[0].index: tq[2], sq2[1].index: tq[3]})

print("After compose_into (explicit qubit_map, qubits 2->3):")
print(f"  Qubits: {target.qubit_count()}, Gates: {target.total_gate_count()}")
print(target.draw())

In [ ]:
# __add__ operator: concatenation into new circuit
circ_a = Circuit()
qa = circ_a.allocate(2)
circ_a.h(qa[0]).cx(qa[0], qa[1])

circ_b = Circuit()
qb = circ_b.allocate(2)
circ_b.x(qb[0]).z(qb[1])

combined = circ_a + circ_b

print(f"circ_a: {circ_a.qubit_count()} qubits, {circ_a.total_gate_count()} gates")
print(f"circ_b: {circ_b.qubit_count()} qubits, {circ_b.total_gate_count()} gates")
print(f"combined: {combined.qubit_count()} qubits, {combined.total_gate_count()} gates")
print()
print(combined.draw())

### Modular Circuit Building

Compose reusable sub-circuits into a larger algorithm.

In [ ]:
def entangling_layer(n: int) -> Circuit:
    """Build a CNOT ladder sub-circuit."""
    circ = Circuit()
    q = circ.allocate(n)
    for i in range(n - 1):
        circ.cx(q[i], q[i + 1])
    return circ

def hadamard_layer(n: int) -> Circuit:
    """Build a Hadamard layer sub-circuit."""
    circ = Circuit()
    q = circ.allocate(n)
    for i in range(n):
        circ.h(q[i])
    return circ

# Build a 4-qubit circuit from reusable layers
n = 4
main = Circuit()
q = main.allocate(n)

main.compose_into(hadamard_layer(n))
main.compose_into(entangling_layer(n))
main.compose_into(hadamard_layer(n))

print(f"Modular circuit: {main.qubit_count()} qubits, {main.total_gate_count()} gates, depth {main.depth()}")
print(main.draw())
print()
print("Generated Q#:")
print(main.to_qsharp())

---

## Notes

- `to_qsharp()` and `to_openqasm()` raise `CodegenError` if the circuit
  contains unbound symbolic parameters — call `bind_parameters()` first.
- `from_qsharp()` supports `use q = Qubit[n];` allocation, standard gates
  (H, X, Y, Z, S, T, CNOT, CZ, SWAP, CCNOT, Rx, Ry, Rz, R1), `Controlled`
  and `Adjoint` modifiers, and `MResetZ`/`M` measurements.
- `from_openqasm()` supports OpenQASM 3.0 with `stdgates.inc`, `qubit[n]`
  declarations, standard gate names, `ctrl @` / `inv @` modifiers, and
  indexed measurement.
- The `+` operator creates fresh qubit allocations in the result —
  the two operands' qubits become disjoint in the combined circuit.
- `compose_into()` reuses the target circuit's qubits — no new allocations.
  The default positional mapping assumes the source has at most as many
  qubits as the target.